In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
file_path = (r"C:\Users\amit6\OneDrive\Desktop\Intelligent-Retail-Customer-Analytics\Data\Processed_data\retail_clean_master.csv")

df = pd.read_csv(
    file_path,
    parse_dates=["InvoiceDate"]
)

df.head()

C:\Users\amit6\AppData\Local\Temp\ipykernel_10504\655991269.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom


In [3]:
print(df.shape)

df.info()

(1007913, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007913 entries, 0 to 1007912
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1007913 non-null  object        
 1   StockCode    1007913 non-null  object        
 2   Description  1007913 non-null  object        
 3   Quantity     1007913 non-null  int64         
 4   InvoiceDate  1007913 non-null  datetime64[ns]
 5   Price        1007913 non-null  float64       
 6   Customer ID  779425 non-null   float64       
 7   Country      1007913 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 61.5+ MB


# SQL Data Preparation

The cleaned retail dataset contains customer, invoice, product, and transaction information in a single table.

For efficient relational database design, the dataset will be normalized into separate entities:

- Customers
- Products
- Invoices
- Transactions

This reduces redundancy, improves data integrity, and enables efficient SQL querying using foreign key relationships.

In [4]:
print("Unique Customers :", df["Customer ID"].nunique())

print("Unique Products :", df["StockCode"].nunique())

print("Unique Invoices :", df["Invoice"].nunique())

print("Total Rows :", len(df))

Unique Customers : 5878
Unique Products : 4917
Unique Invoices : 40079
Total Rows : 1007913


In [7]:
customers = (
    df[["Customer ID", "Country"]]
    .dropna(subset=["Customer ID"])
    .drop_duplicates()
    .sort_values("Customer ID")
    .reset_index(drop=True)
)
customers.head()

,Customer ID,Country
0,12346.00,United Kingdom
1,12347.00,Iceland
2,12348.00,Finland
3,12349.00,Italy
4,12350.00,Norway


In [8]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5891 entries, 0 to 5890
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer ID  5891 non-null   float64
 1   Country      5891 non-null   object 
dtypes: float64(1), object(1)
memory usage: 92.2+ KB


In [9]:
customers["Customer ID"].duplicated().sum()

np.int64(13)

In [10]:
duplicate_customers = (
    customers[customers["Customer ID"].duplicated(keep=False)]
    .sort_values("Customer ID")
)

duplicate_customers

,Customer ID,Country
24,12370.00,Austria
25,12370.00,Cyprus
48,12394.00,Denmark
49,12394.00,Belgium
68,12413.00,Spain
69,12413.00,France
73,12417.00,Spain
74,12417.00,Belgium
79,12422.00,Australia
80,12422.00,Switzerland


In [11]:
duplicate_customers.groupby("Customer ID").agg({
    "Country": lambda x: list(x.unique())
})

,Country
Customer ID,
12370.00,"[Austria, Cyprus]"
12394.00,"[Denmark, Belgium]"
12413.00,"[Spain, France]"
12417.00,"[Spain, Belgium]"
12422.00,"[Australia, Switzerland]"
12423.00,"[Belgium, Denmark]"
12429.00,"[Austria, Denmark]"
12431.00,"[Australia, Belgium]"
12449.00,"[Denmark, Belgium]"


In [12]:
duplicate_ids = duplicate_customers["Customer ID"].unique()

for cid in duplicate_ids:
    print(f"\nCustomer ID: {cid}")
    print(df[df["Customer ID"] == cid]["Country"].value_counts())


Customer ID: 12370.0
Country
Cyprus     158
Austria     43
Name: count, dtype: int64

Customer ID: 12394.0
Country
Belgium    21
Denmark     6
Name: count, dtype: int64

Customer ID: 12413.0
Country
France    38
Spain     13
Name: count, dtype: int64

Customer ID: 12417.0
Country
Belgium    335
Spain       34
Name: count, dtype: int64

Customer ID: 12422.0
Country
Switzerland    113
Australia       59
Name: count, dtype: int64

Customer ID: 12423.0
Country
Belgium    152
Denmark     13
Name: count, dtype: int64

Customer ID: 12429.0
Country
Denmark    132
Austria     21
Name: count, dtype: int64

Customer ID: 12431.0
Country
Australia    247
Belgium      143
Name: count, dtype: int64

Customer ID: 12449.0
Country
Belgium    191
Denmark     10
Name: count, dtype: int64

Customer ID: 12455.0
Country
Cyprus    179
Spain      48
Name: count, dtype: int64

Customer ID: 12457.0
Country
Switzerland    82
Cyprus          2
Name: count, dtype: int64

Customer ID: 12652.0
Country
France     48


In [17]:
customers = (
    df.groupby("Customer ID", as_index=False)
      .agg({
          "Country": lambda x: x.mode().iloc[0]
      })
)

customers.head()

,Customer ID,Country
0,12346.00,United Kingdom
1,12347.00,Iceland
2,12348.00,Finland
3,12349.00,Italy
4,12350.00,Norway


In [30]:
customers.to_csv(
    "customers.csv",
    index=False
)

### Handling Duplicate Customer Countries

During database normalization, 13 Customer IDs were associated with more than one country.

To maintain CustomerID as the primary key, each customer was assigned the country that appeared most frequently across their transaction history (mode).

This approach preserves one unique customer record while minimizing the impact of inconsistent source data.

In [20]:
product_check = (
    df.groupby("StockCode")
      .agg(
          Description_Count=("Description", "nunique"),
          Price_Count=("Price", "nunique")
      )
)

product_check.head()

,Description_Count,Price_Count
StockCode,,
10002,1,5
10002R,1,2
10080,1,2
10109,1,1
10120,1,2


In [22]:
product_check.shape

(4917, 2)

In [23]:
description_issue = product_check[
    product_check["Description_Count"] > 1
]

print("Products with multiple descriptions:", len(description_issue))

Products with multiple descriptions: 648


In [24]:
price_issue = product_check[
    product_check["Price_Count"] > 1
]

print("Products with multiple prices:", len(price_issue))

Products with multiple prices: 4358


In [25]:
description_issue.index[:10]

Index(['15058A', '15058B', '16011', '16012', '16151A', '16156L', '16161U',
       '17107D', '17129F', '18094C'],
      dtype='object', name='StockCode')

In [26]:
for code in description_issue.index[:10]:
    print(f"\nStockCode: {code}")
    print(
        df[df["StockCode"] == code][["Description"]]
        .drop_duplicates()
    )


StockCode: 15058A
                            Description
64540   BLUE WHITE SPOTS GARDEN PARASOL
505639     BLUE POLKADOT GARDEN PARASOL

StockCode: 15058B
                            Description
52755   PINK WHITE SPOTS GARDEN PARASOL
442585     PINK POLKADOT GARDEN PARASOL

StockCode: 16011
             Description
10159    ANIMAL STICKERS
114700   ANIMAL STICKERS

StockCode: 16012
                       Description
3966    FOOD/DRINK SPUNGE STICKERS
301744  FOOD/DRINK SPONGE STICKERS

StockCode: 16151A
                                Description
174271  FLOWER DES BLUE HANDBAG/ORANG HANDL
662274      FLOWERS HANDBAG blue and orange

StockCode: 16156L
           Description
47298   WRAP, CAROUSEL
616663   WRAP CAROUSEL

StockCode: 16161U
                  Description
2373    WRAP,SUKI AND FRIENDS
247983  WRAP SUKI AND FRIENDS

StockCode: 17107D
                                Description
9950    FLOWER FAIRY,5 SUMMER B'DRAW LINERS
788864         FLOWER FAIRY 5 DRAWER LINERS
820024 

In [27]:
products = (
    df.groupby("StockCode", as_index=False)
      .agg({
          "Description": lambda x: x.mode().iloc[0]
      })
)

products.head()

,StockCode,Description
0,10002,INFLATABLE POLITICAL GLOBE
1,10002R,ROBOT PENCIL SHARPNER
2,10080,GROOVY CACTUS INFLATABLE
3,10109,BENDY COLOUR PENCILS
4,10120,DOGGY RUBBER


In [28]:
print("Rows:", len(products))

print("Unique StockCodes:", products["StockCode"].nunique())

print("Duplicate StockCodes:", products["StockCode"].duplicated().sum())

Rows: 4917
Unique StockCodes: 4917
Duplicate StockCodes: 0


In [29]:
products.to_csv(
    "products.csv",
    index=False
)

### Handling Product Master Data

During normalization, 648 StockCodes were associated with multiple product descriptions due to spelling variations, formatting differences, and inconsistent data entry.

To create a single product master record, the most frequently occurring description (mode) was selected for each StockCode.

The `Price` column was intentionally excluded from the Products table because product prices vary across transactions due to promotions, discounts, and historical price changes. Price will instead be stored in the Transactions table to preserve transaction-level pricing.

In [31]:
invoice_customer_check = (
    df.groupby("Invoice")
      .agg(Customer_Count=("Customer ID", "nunique"))
)

invoice_customer_issue = invoice_customer_check[
    invoice_customer_check["Customer_Count"] > 1
]

print("Invoices with multiple customers:", len(invoice_customer_issue))

Invoices with multiple customers: 0


In [32]:
invoice_date_check = (
    df.groupby("Invoice")
      .agg(Date_Count=("InvoiceDate", "nunique"))
)

invoice_date_issue = invoice_date_check[
    invoice_date_check["Date_Count"] > 1
]

print("Invoices with multiple dates:", len(invoice_date_issue))

Invoices with multiple dates: 82


In [34]:
invoice_date_issue.index[:10]
sample_invoice = invoice_date_issue.index[0]

df[df["Invoice"] == sample_invoice].sort_values("InvoiceDate")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
40137,492807,85131A,BEADED PEARL HEART WHITE ON STICK,1,2009-12-20 12:28:00,1.25,17211.00,United Kingdom
40161,492807,22112,CHOCOLATE HOT WATER BOTTLE,3,2009-12-20 12:28:00,4.95,17211.00,United Kingdom
40160,492807,84508B,STRIPES DESIGN TEDDY,2,2009-12-20 12:28:00,2.55,17211.00,United Kingdom
40159,492807,84327A,PINK JUMPER LARRY THE LAMB,2,2009-12-20 12:28:00,2.10,17211.00,United Kingdom
40158,492807,84507B,STRIPES DESIGN MONKEY DOLL,3,2009-12-20 12:28:00,2.55,17211.00,United Kingdom
40157,492807,84507C,BLUE CIRCLES DESIGN MONKEY DOLL,3,2009-12-20 12:28:00,2.55,17211.00,United Kingdom
40156,492807,84508C,BLUE CIRCLES DESIGN TEDDY,2,2009-12-20 12:28:00,2.55,17211.00,United Kingdom
40155,492807,21497,FANCY FONTS BIRTHDAY WRAP,25,2009-12-20 12:28:00,0.42,17211.00,United Kingdom
40154,492807,22296,HEART IVORY TRELLIS LARGE,6,2009-12-20 12:28:00,1.65,17211.00,United Kingdom
40153,492807,22294,HEART FILIGREE DOVE SMALL,6,2009-12-20 12:28:00,1.25,17211.00,United Kingdom


In [35]:
invoices = (
    df.groupby("Invoice", as_index=False)
      .agg(
          CustomerID=("Customer ID", "first"),
          InvoiceDate=("InvoiceDate", "min")
      )
)

In [36]:
print("Rows:", len(invoices))
print("Unique Invoices:", invoices["Invoice"].nunique())
print("Duplicate Invoices:", invoices["Invoice"].duplicated().sum())
print("Null Customer IDs:", invoices["CustomerID"].isna().sum())

Rows: 40079
Unique Invoices: 40079
Duplicate Invoices: 0
Null Customer IDs: 3108


In [38]:
print("Rows:", len(invoices))
print("Duplicate Invoice:", invoices["Invoice"].duplicated().sum())
print("Null Customer IDs:", invoices["CustomerID"].isna().sum())

Rows: 40079
Duplicate Invoice: 0
Null Customer IDs: 3108


In [39]:
invoices.to_csv(
    "invoices.csv",
    index=False
)

### Handling Missing Customer IDs

A total of 3,108 invoices were associated with missing Customer IDs. These invoices represent valid retail transactions where the customer identity was unavailable (e.g., guest purchases or missing CRM information).

These records were retained to preserve transaction history. In the relational database, `CustomerID` is stored as a nullable foreign key in the `Invoices` table.

In [40]:
transactions = df[[
    "Invoice",
    "StockCode",
    "Quantity",
    "Price"
]].copy()

In [41]:
transactions["TotalAmount"] = (
    transactions["Quantity"] *
    transactions["Price"]
)

In [ ]:
transactions.insert(
    0,
    "TransactionID",
    range(1, len(transactions) + 1)
)



ValueError: cannot insert TransactionID, already exists

In [47]:
print("Rows:", len(transactions))
print("Duplicate Transaction IDs:", transactions["TransactionID"].duplicated().sum())
print("Null Invoice Numbers:", transactions["Invoice"].isna().sum())
print("Null Stock Codes:", transactions["StockCode"].isna().sum())

Rows: 1007913
Duplicate Transaction IDs: 0
Null Invoice Numbers: 0
Null Stock Codes: 0


In [48]:
missing_invoice = (
    ~transactions["Invoice"].isin(invoices["Invoice"])
).sum()

print("Transactions with missing Invoice:", missing_invoice)

Transactions with missing Invoice: 0


In [49]:
missing_product = (
    ~transactions["StockCode"].isin(products["StockCode"])
).sum()

print("Transactions with missing Product:", missing_product)

Transactions with missing Product: 0


In [54]:
transactions = df[
    ["Invoice", "Customer ID", "StockCode", "Quantity", "Price"]
].copy()

transactions.rename(columns={
    "Invoice": "InvoiceNo",
    "Customer ID": "CustomerID",
    "Price": "UnitPrice"
}, inplace=True)

transactions["TotalAmount"] = (
    transactions["Quantity"] *
    transactions["UnitPrice"]
)

transactions.insert(
    0,
    "TransactionID",
    range(1, len(transactions) + 1)
)

In [57]:
print("Missing Customer IDs in df:", df["Customer ID"].isna().sum())

print("Unique invoices with missing Customer ID:",
      df[df["Customer ID"].isna()]["Invoice"].nunique())

print("Total rows in df:", len(df))

Missing Customer IDs in df: 228488
Unique invoices with missing Customer ID: 3108
Total rows in df: 1007913


In [60]:
transactions = transactions.dropna(subset=["CustomerID"]).copy()

In [61]:
transactions.to_csv(
    "transactions.csv",
    index=False
)

In [56]:
transactions["CustomerID"].isna().sum()

np.int64(228488)

### Creating the Transactions Table

The Transactions table was created directly from the cleaned retail dataset. Each row represents a single product purchased within an invoice.

A unique `TransactionID` was generated for every record. Product prices were retained at the transaction level to preserve historical pricing information and avoid redundancy in the Products table.

Referential integrity was validated by confirming that every `InvoiceNo` exists in the Invoices table and every `StockCode` exists in the Products table.

In [52]:
print("Rows:", len(transactions))
print("Duplicate Transaction IDs:", transactions["TransactionID"].duplicated().sum())
print("Missing Invoice References:", (~transactions["Invoice"].isin(invoices["Invoice"])).sum())
print("Missing Product References:", (~transactions["StockCode"].isin(products["StockCode"])).sum())

Rows: 1007913
Duplicate Transaction IDs: 0
Missing Invoice References: 0
Missing Product References: 0


In [53]:
invoices_sql = invoices.copy()

# Replace NaN with empty string
invoices_sql["CustomerID"] = invoices_sql["CustomerID"].fillna("")

# Save a separate file for SQL import
invoices_sql.to_csv(
    r"invoices_sql.csv",
    index=False
)